# Transformación · Silver

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.window import Window
from datetime import datetime, timezone
import csv
import re
DEFAULTS = {'storageName': 'adlsmardata1412jmvcc', 'catalogo': 'fi_videojuegos', 'storageCredential': 'credential', 'container_metastore': 'metastore', 'carpeta_proyecto': 'fi_videojuegos_jmcc', 'raw_subdir': '', 'aplicar_grants': 'false'}
for clave, valor in DEFAULTS.items():
    dbutils.widgets.text(clave, valor)
PARAMS = {clave: dbutils.widgets.get(clave) for clave in DEFAULTS}
CATALOGO = PARAMS['catalogo'].strip()
STORAGE = PARAMS['storageName'].strip()
CARPETA_PROYECTO = PARAMS['carpeta_proyecto'].strip('/')
raw_subdir = PARAMS['raw_subdir'].strip('/')
assert re.fullmatch('[a-z][a-z0-9_]*', CATALOGO)
assert re.fullmatch('[a-z0-9]{3,24}', STORAGE)
assert re.fullmatch('[a-z0-9_/-]+', CARPETA_PROYECTO) and '..' not in CARPETA_PROYECTO
assert not raw_subdir or (re.fullmatch('[A-Za-z0-9_/-]+', raw_subdir) and '..' not in raw_subdir)
RUTA_RAW = f'abfss://raw@{STORAGE}.dfs.core.windows.net/' + raw_subdir
RUTA_RAW = RUTA_RAW.rstrip('/')
TABLAS_EXTERNAS = True

def q(s):
    return '`' + s.replace('`', '``') + '`'

def nombre(capa, tabla):
    assert capa in ('bronze', 'silver', 'golden')
    assert tabla.startswith('fi_'), 'Solo se permiten tablas de este proyecto.'
    return f'{q(CATALOGO)}.{q(capa)}.{q(tabla)}'

def ruta_delta(capa, tabla):
    return f'abfss://{capa}@{STORAGE}.dfs.core.windows.net/{CARPETA_PROYECTO}/{tabla}'

def preparar(df, capa, tabla):
    destino = nombre(capa, tabla)
    actual = spark.table(destino)
    assert [(f.name, f.dataType) for f in actual.schema] == [(f.name, f.dataType) for f in df.schema], f'Esquema diferente en {destino}. Revisar antes de reemplazar datos.'
    detalle = spark.sql(f'DESCRIBE DETAIL {destino}').first().asDict()
    assert detalle['format'].lower() == 'delta'
    if TABLAS_EXTERNAS:
        assert detalle['location'].rstrip('/') == ruta_delta(capa, tabla).rstrip('/'), f'Tabla ya registrada en otra ruta: {destino}. No se modifica.'
    else:
        tipo = spark.sql(f'DESCRIBE TABLE EXTENDED {destino}').collect()
        tipos = [r['data_type'] for r in tipo if r['col_name'].strip() == 'Type']
        assert tipos and tipos[0].upper() == 'MANAGED', f'{destino} ya es externa.'
    return destino

def guardar(df, capa, tabla):
    destino = preparar(df, capa, tabla)
    df.select(*spark.table(destino).columns).write.mode('overwrite').insertInto(destino)
    resultado = spark.table(destino)
    assert resultado.count() == df.count(), f'Conteo diferente: {destino}'
    print('OK', destino, resultado.count(), 'filas')
    return resultado

## Dos tablas Silver
Ventas tipadas permanece como DataFrame intermedio. Se guardan metadatos y ventas enriquecidas; se conserva la lógica de cruce y sus validaciones.

## S02 — Funciones sencillas de limpieza
try_cast conserva como NULL los valores no convertibles. No se promedian escalas de distintas fuentes.

In [0]:
def texto(c):
    valor = F.trim(F.col(c))
    return F.when(F.length(valor) > 0, valor)

def convertir(c, tipo):
    return F.expr(f"try_cast({q(c)} AS {tipo})")

def normalizar(c):
    valor = F.lower(F.trim(F.coalesce(c, F.lit(""))))
    valor = F.translate(valor, "áàäâéèëêíìïîóòöôúùüûñ", "aaaaeeeeiiiioooouuuun")
    return F.trim(F.regexp_replace(valor, "[^a-z0-9]+", " "))

def huella(df):
    cols = sorted(c for c in df.columns if not c.startswith("fi_"))
    return F.sha2(F.to_json(F.struct(*[F.col(q(c)).alias(c) for c in cols]),
                           options={"ignoreNullFields":"false"}), 256)

def origen(tabla):
    return spark.table(nombre("bronze", tabla))

def familia(c):
    return (F.when(c.isin("NES","SNES","N64","GC","Wii","WiiU","DS","3DS","GB","GBA"), "Nintendo")
             .when(c.isin("PS","PS2","PS3","PS4","PSP","PSV"), "PlayStation")
             .when(c.isin("XB","X360","XOne"), "Xbox").otherwise("Otras plataformas"))

## S03 — Leer VG Sales y convertir tipos
Una fila por Rank. Las ventas están en millones de unidades estimadas; no son ingresos.

In [0]:
br_ventas = origen("fi_br_vgsales")
ventas = br_ventas.select(
    convertir("Rank", "int").alias("rank_venta"), texto("Name").alias("titulo"),
    texto("Platform").alias("plataforma"), convertir("Year", "int").alias("anio"),
    texto("Genre").alias("genero"), texto("Publisher").alias("editorial"),
    *[convertir(c, "decimal(14,3)").alias(n) for c,n in [
      ("NA_Sales","ventas_na"),("EU_Sales","ventas_eu"),("JP_Sales","ventas_jp"),
      ("Other_Sales","ventas_otras"),("Global_Sales","ventas_globales")]],
    F.col("fi_archivo_origen"), F.col("fi_fecha_ingesta"))
assert ventas.count() == br_ventas.count() == 16598
assert ventas.where("rank_venta IS NULL").count() == 0
assert ventas.select("rank_venta").distinct().count() == ventas.count()

## S04 — Normalizar títulos y clasificar ventas
Las ventas de otras plataformas se conservan. Las diferencias de suma regional se marcan sin borrar filas.

In [0]:
ventas = (ventas.withColumn("titulo_clave", normalizar(F.col("titulo")))
          .withColumn("familia", familia(F.col("plataforma")))
          .withColumn("venta_valida", F.coalesce((F.length("titulo_clave") > 0) &
              F.col("ventas_globales").isNotNull() & (F.col("ventas_globales") >= 0), F.lit(False)))
          .withColumn("diferencia_regiones", F.abs(F.col("ventas_globales") - (
              F.col("ventas_na") + F.col("ventas_eu") + F.col("ventas_jp") + F.col("ventas_otras")))))
ventas = ventas.withColumn("regiones_conciliadas", F.col("diferencia_regiones") <= F.lit(0.03))
display(ventas.limit(5))

## S06 — Preparar Nintendo Metacritic
No hay CSV IGDB de Nintendo. Switch y plataformas sin código se conservan, pero no se fuerza una coincidencia con ventas.

In [0]:
br_nintendo = origen("fi_br_nintendo")
nintendo = br_nintendo.select(
    F.lit("Metacritic").alias("fuente"), F.lit("Nintendo").alias("familia"),
    texto("game_title").alias("titulo_meta"), texto("platform").alias("nombre_plataforma"),
    texto("platform_vgsales_code").alias("plataforma"),
    convertir("release_year","int").alias("anio_meta"),
    convertir("release_date","date").alias("fecha_meta"),
    convertir("meta_score","double").alias("meta_score"),
    convertir("user_score","double").alias("user_score"),
    texto("developer_1").alias("desarrollador"), texto("genres_original").alias("generos_meta"),
    F.lit(None).cast("long").alias("id_igdb"),
    F.lit(None).cast("double").alias("score_igdb"),
    F.lit(None).cast("double").alias("score_hltb"),
    F.lit(None).cast("double").alias("horas_principal"),
    F.lit(None).cast("double").alias("horas_extras"),
    F.lit(None).cast("double").alias("horas_completista"),
    F.col("fi_archivo_origen"), huella(br_nintendo).alias("hash_origen"))
nintendo = nintendo.withColumn("titulo_clave", normalizar(F.col("titulo_meta")))

## S07 — Función IGDB/HLTB
PlayStation y Xbox comparten encabezado. Mantiene puntuaciones y horas separadas de Metacritic.

In [0]:
def transformar_igdb(tabla, marca):
    bruto = origen(tabla)
    return bruto.select(
        F.lit("IGDB_HLTB").alias("fuente"), F.lit(marca).alias("familia"),
        texto("nombre_videojuego").alias("titulo_meta"), texto("plataforma").alias("nombre_plataforma"),
        texto("codigo_plataforma_vgsales").alias("plataforma"),
        convertir("anio_primer_lanzamiento","int").alias("anio_meta"),
        convertir("fecha_primer_lanzamiento","date").alias("fecha_meta"),
        F.lit(None).cast("double").alias("meta_score"),
        F.lit(None).cast("double").alias("user_score"),
        F.lit(None).cast("string").alias("desarrollador"), texto("generos").alias("generos_meta"),
        convertir("id_igdb","long").alias("id_igdb"),
        convertir("calificacion_igdb_0_100","double").alias("score_igdb"),
        convertir("puntuacion_hltb","double").alias("score_hltb"),
        convertir("horas_historia_principal","double").alias("horas_principal"),
        convertir("horas_historia_mas_extras","double").alias("horas_extras"),
        convertir("horas_completista_100","double").alias("horas_completista"),
        F.col("fi_archivo_origen"), huella(bruto).alias("hash_origen")
    ).withColumn("titulo_clave", normalizar(F.col("titulo_meta")))

## S08 — Transformar PlayStation
Una fuente por celda.

In [0]:
playstation = transformar_igdb("fi_br_playstation", "PlayStation")
assert playstation.count() == 31002

## S09 — Transformar Xbox
Una fuente por celda.

In [0]:
xbox = transformar_igdb("fi_br_xbox", "Xbox")
assert xbox.count() == 17993

## S10 — Consolidar metadatos
Se conservan las 50.021 filas; los duplicados se identifican al elegir candidatos, no se borran aquí.

In [0]:
metadatos = nintendo.unionByName(playstation).unionByName(xbox)
assert metadatos.count() == 50021
metadatos = metadatos.withColumn("apto_join", (F.length("titulo_clave") > 0) & F.col("plataforma").isNotNull())

## S11 — Validar escalas y horas
Valores fuera de rango se convierten a NULL en Silver. Se añade una bandera de corrección; Bronze preserva los originales.

In [0]:
condiciones = [F.col("meta_score").isNotNull() & ~F.col("meta_score").between(0,100),
               F.col("user_score").isNotNull() & ~F.col("user_score").between(0,10),
               F.col("score_igdb").isNotNull() & ~F.col("score_igdb").between(0,100)]
metadatos = metadatos.withColumn("score_fuera_rango", condiciones[0] | condiciones[1] | condiciones[2])
for c, maximo in [("meta_score",100),("user_score",10),("score_igdb",100)]:
    metadatos = metadatos.withColumn(c, F.when(F.col(c).between(0,maximo),F.col(c)))
for c in ["horas_principal","horas_extras","horas_completista"]:
    metadatos = metadatos.withColumn(c, F.when(F.col(c) >= 0,F.col(c)))
metadatos = guardar(metadatos, "silver", "fi_sl_metadatos")

## S12 — Preparar candidatos de coincidencia
Regla explícita: título normalizado + código de plataforma + año compatible (±3). Si falta año se permite, con confianza menor. Sin coincidencia se conserva la venta.

In [0]:
TOLERANCIA_ANIOS = 3
# Duplicados exactamente iguales de metadatos no aumentan candidatos.
md_join = metadatos.where("apto_join").dropDuplicates(["hash_origen"])
v = ventas.alias("v")
m = md_join.alias("m")
delta = F.abs(F.col("v.anio") - F.col("m.anio_meta"))
condicion = ((F.col("v.titulo_clave") == F.col("m.titulo_clave")) &
             (F.col("v.plataforma") == F.col("m.plataforma")) &
             (F.col("v.anio").isNull() | F.col("m.anio_meta").isNull() | (delta <= TOLERANCIA_ANIOS)))
candidatos = v.join(m, condicion, "left")

## S13 — Elegir un candidato por venta
Orden determinista: año conocido, menor diferencia, puntuación disponible y hash. No se usa el título solo ni se multiplican las ventas.

In [0]:
particion = Window.partitionBy(F.col("v.rank_venta"))
orden = particion.orderBy(
    F.when(F.col("m.hash_origen").isNull(),2)
     .when(F.col("v.anio").isNull() | F.col("m.anio_meta").isNull(),1).otherwise(0),
    delta.asc_nulls_last(),
    F.coalesce(F.col("m.score_igdb"), F.col("m.meta_score")).desc_nulls_last(),
    F.col("m.hash_origen").asc_nulls_last())
elegidos = (candidatos.withColumn("fi_num_candidatos", F.count(F.col("m.hash_origen")).over(particion))
            .withColumn("fi_orden", F.row_number().over(orden)).where("fi_orden = 1"))
campos_meta = ["fuente","titulo_meta","nombre_plataforma","anio_meta","fecha_meta","meta_score", "user_score",
               "desarrollador","generos_meta","id_igdb","score_igdb","score_hltb",
               "horas_principal","horas_extras","horas_completista","hash_origen"]
enriquecidas = elegidos.select(
    *[F.col(f"v.{q(c)}").alias(c) for c in ventas.columns],
    *[F.col(f"m.{q(c)}").alias(c) for c in campos_meta], F.col("fi_num_candidatos"))

## S14 — Marcar confianza y guardar
La conciliación comprueba todas las ventas, incluso otras plataformas y registros sin metadatos.

In [0]:
enriquecidas = enriquecidas.withColumn("confianza_join",
    F.when(F.col("hash_origen").isNull(),"SIN_COINCIDENCIA")
     .when(F.col("anio").isNull() | F.col("anio_meta").isNull(),"BAJA_SIN_ANIO")
     .when(F.col("anio") == F.col("anio_meta"),"ALTA_ANIO_IGUAL").otherwise("MEDIA_DIF_ANIO"))
assert enriquecidas.count() == ventas.count() == 16598
assert enriquecidas.select("rank_venta").distinct().count() == 16598
assert enriquecidas.agg(F.sum("ventas_globales")).first()[0] == ventas.agg(F.sum("ventas_globales")).first()[0]
enriquecidas = guardar(enriquecidas, "silver", "fi_sl_ventas_enriquecidas")

## S15 — Evidencia Silver
La cobertura es informativa: tener cero coincidencias en una plataforma no justifica borrar sus ventas.

In [0]:
display(enriquecidas.groupBy("familia","confianza_join").count().orderBy("familia","confianza_join"))
display(enriquecidas.agg(F.count("*").alias("ventas"),
    F.sum(F.when(F.col("fi_num_candidatos") > 1,1).otherwise(0)).alias("ventas_con_varios_candidatos")))
print("SILVER COMPLETADO. Sigue con 04_Load_Golden.")